In [0]:
# ============================================================
# DOWNLOAD - GERAÇÃO DE USINAS ONS
# ============================================================
import requests
import os
from datetime import date

# ------------------------------------------------------------
# CONFIGURAÇÕES
# ------------------------------------------------------------
ano_inicio = 2022
mes_inicio = 1

ano_fim = 2026
mes_fim = 9

# Volume STAGE
caminho_stage = "/Volumes/mba/stage/dados_bruto/usinas/GERACAO_USINA"

# URL base oficial dos arquivos Parquet do ONS
url_base = (
    "https://ons-aws-prod-opendata.s3.amazonaws.com/"
    "dataset/geracao_usina_2_ho/"
)

# ------------------------------------------------------------
# CRIA DIRETÓRIO
# ------------------------------------------------------------
os.makedirs(caminho_stage, exist_ok=True)


# ------------------------------------------------------------
# DOWNLOAD DOS ARQUIVOS
# ------------------------------------------------------------
for ano in range(ano_inicio, ano_fim + 1):
    mes_ini = mes_inicio if ano == ano_inicio else 1
    mes_fim = mes_fim if ano == ano_fim else 12

    for mes in range(mes_ini, mes_fim + 1):
        periodo = f"{ano}_{mes:02d}"
        nome_arquivo = f"GERACAO_USINA-2_{periodo}.parquet"
        url = url_base + nome_arquivo

        caminho_destino = os.path.join(
            caminho_stage,
            nome_arquivo
        )

        # Evita baixar novamente
        if os.path.exists(caminho_destino):
            print(f"JÁ EXISTE: {nome_arquivo}")
            continue

        print(f"Baixando: {nome_arquivo}")

        try:
            response = requests.get(
                url,
                timeout=300
            )
            response.raise_for_status()

            with open(caminho_destino, "wb") as arquivo:
                arquivo.write(response.content)

            tamanho_mb = len(response.content) / 1024 / 1024

            print(
                f"OK - {nome_arquivo} "
                f"({tamanho_mb:.2f} MB)"
            )

        except Exception as e:
            print(
                f"ERRO - {nome_arquivo}: {e}"
            )

In [0]:
dbutils.notebook.exit("OK")

In [0]:
df_geracao = spark.read.parquet(
    "dbfs:/Volumes/mba/stage/dados_bruto/ONS/GERACAO_USINA/*.parquet"
)

display(df_geracao)